In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# PT-W3-D7：LangChat × Ontology × Compiler 集成方案 — "语义操作系统安装到哪一层"

> 📅 Week 3 - Day 7 | 2026-08-16
>
> **并行轨道：Business Semantic Architecture · Week 3 毕业输出日**
>
> **今日主题：输出《LangChat × Ontology × Compiler 集成方案 v0.1》——Ontology 与编译器在你的架构里到底放在哪里**

---


## 一句话开篇

> **集成方案的核心不是"再加一个新组件"，而是回答一个问题：从"业务世界"到"Agent 执行"的语义链路里，每一段的权威在哪、产物是什么、边界在哪。答案你已经写进了三套 ADR——今天只是把它显式画成一张图。**

---


## 二、先回顾：这条链路你其实已经建了大半

Week 3 六天走过来，链路逐渐清晰：


In [ ]:
Day 1: Agent 利用 Ontology        → Agent 需要的不是数据，是"世界模型 + 规则 + 许可"
Day 2: Ontology Compiler          → BCM ADR-001/005 已经是 Semantic Compiler 的原则章程
Day 3: Bounded Context 约束认知    → Context 边界 = Agent 认知边界（防幻觉泛滥）
Day 4: Policy 约束执行            → effect_policy / required_scopes = "知道 ≠ 可以"
Day 5: Digital Employee 语义定义   → 语义锚点收束为角色定义（指向前，不包含）
Day 6: 三套 ADR 在 Ontology 层统一 → 世界模型层 / 语义治理层 / 执行架构层


今天的任务：把这六天收束成**一份可执行的集成方案**。

---


## 三、集成方案 v0.1：全景语义链路

### 3.1 四层架构（权威归位）


In [ ]:
┌───────────────────────────────────────────────────────┐
│ L1 世界模型层（World Model）—— MI Domain Model v1.0     │
│    权威：17 个 Bounded Context / Object Ownership /     │
│    D-014 商业事实唯一来源 / Lifecycle 状态定义           │
│    回答：业务世界里有什么？谁拥有？身份是什么？            │
├───────────────────────────────────────────────────────┤
│ L2 语义治理层（Semantic Governance）—— CRE BCM ADR-001~006│
│    权威：Meta Object 分类（002）/ Ownership（003）/       │
│    Binding 四值（004）/ 编译器五职责（005）/             │
│    四类关系 + effect_type 五类（006，Published 冻结）      │
│    回答：这些实体如何组成能力？关系如何归类？              │
│          生命周期变化如何传播？什么算合法组合？            │
├───────────────────────────────────────────────────────┤
│ L3 编译层（Semantic Compiler）—— BCM ADR-005 职责封闭集  │
│    职责：Validate / Resolve / Assemble / Governance /    │
│    Artifact —— 输出 Minimum Design Artifact（八字段）    │
│    回答：一份业务组合声明，如何变成结构化的设计制品？       │
├───────────────────────────────────────────────────────┤
│ L4 执行层（Agent Execution）—— LangChat ADR-001~009      │
│    权威：Blueprint / ApplicationContract / SkillRelease  │
│    v2 / DigitalEmployeeDefinition / DeploymentRevision   │
│    回答：设计制品如何物化为 Agent 可部署、可调用、         │
│    受 Policy 约束的运行时对象？                          │
└───────────────────────────────────────────────────────┘
        ↕ 底座：MI 307 表 + Go 模块化单体 = 事实的最终载体


**关键认知：这四层不是四个系统，是一条语义供应链。** 上游供给语义，下游消费语义；每层只有一个权威（D-014 原则的层间放大）。

### 3.2 语义编译流水线（端到端）

用一个真实场景走通全链路——**"新商户入驻"数字员工技能的诞生**：

| 阶段 | 发生什么 | 权威文档 | 产物 |
|------|---------|---------|------|
| ① 描述（Describe） | 业务方声明："招商域需要一个'铺位可租判断'能力，涉及 Space/Lease/Tenant 三个对象" | BCM Master Matrix + 01-招商管理.md | capability 行（`CRE-*`）+ 对象引用 |
| ② 治理校验（Validate） | 编译器检查：对象是否在 Meta Object 分类内？Ownership 是否有唯一 canonical owner？Binding 类型是否在四值内？ | ADR-002/003/004 | 校验通过 / 驳回清单 |
| ③ 组装（Assemble） | 按 Typed Binding 组合对象与能力，声明 effect（如 Lease 终止 → release Space，前置 inspection） | ADR-005 + effect-registry.yaml | **Minimum Design Artifact**（八字段） |
| ④ 物化（Materialize） | LangChat 侧把 Design Artifact 物化为 Blueprint → ApplicationContract → SkillRelease | LangChat ADR-005/003 | 可部署技能制品链 |
| ⑤ 执行（Execute） | 数字员工运行时消费语义：Context 认知边界（D3）+ effect_policy 约束（D4）+ MCP Tool 调用 MI | LangChat ADR-006/007/008 | 受控的 Agent 行为 |

**这条流水线就是你说的 `Business Model → Semantic Model → Capability → SkillRelease → ExecutionPlanIR`，只是现在每一站的输入、输出、权威都有 ADR 背书。**

---


## 四、五个关键放置决策（集成方案的骨架）

集成方案不是画完图就结束，要给出**放置决策**——每个构件放哪层、归谁管：

| # | 决策 | 内容 | 依据 |
|---|------|------|------|
| P1 | **Ontology 词汇表放 L2，不放 LangChat** | `CRE-*` 行、effect_type 五类、binding_type 四值、四类关系——业务语义的"应当存在"由 BCM 定义，产品只核验"是否做到" | HC-1：矩阵是业务侧上游权威 |
| P2 | **编译器只出设计制品，不出运行时对象** | 编译器五职责封闭集的产物是 Design Artifact；物化成什么（Blueprint？Skill？）决策权归 LangChat | ADR-005 §4.3：产物物化决策权归下游 |
| P3 | **Agent 认知边界 = Bounded Context 边界** | 数字员工的"知识范围"用 Context 划定，不用技术模块划定；跨 Context 走受控引用（Context Map 契约），不许 Agent 自由 join | D3 + MI Domain Model §4 |
| P4 | **Policy 双层声明** | 业务侧（BCM）：什么条件下允许什么 effect（语义层）；执行侧（LangChat）：required_scopes / effect_policy（运行时强制）。两层通过 effect_type 对齐，不重复定义 | D4 + ADR-006 + LangChat ADR-005 |
| P5 | **事实回流单向** | 运行时产生的业务事实（如"Lease #123 已终止"）只回流 L1（MI 是唯一事实 Owner），L2/L3/L4 只存类型与规则，不存实例 | D-1 业务事实与运行时分离 + D-014 |

> 这五条决策合起来就是一句话：**"类型在上游定义，事实在底座落地，中间只编译语义，不复制权威。"**

---


## 五、对 AI Agent 的意义：装上"语义操作系统"前后对比

回到根本问题——**这套集成如何帮助 AI Agent 理解企业？**

| 能力维度 | 没有集成方案 | 有集成方案 |
|---------|-------------|-----------|
| 世界认知 | 靠读 PRD + 表结构，自行猜测"铺位可租"是什么意思 | Context + Object Ownership 给出权威实体与身份 |
| 关系推理 | 只见外键，"Lease 终止影响什么"要翻代码 | 四类关系 + effect_type：`Lease --terminates--> release Space (requires inspection)` |
| 行为边界 | "能做多少做多少"或靠 prompt 约束 | 认知边界（Context）+ 执行边界（Policy）双重声明 |
| 技能获取 | 每个技能手工写 prompt + 手工接工具 | 能力行 → 编译 → SkillRelease，语义即技能的出生证明 |
| 演进安全 | 业务规则变了，Agent 行为悄悄漂移 | 受控词表冻结（ORE-1）+ ADR 修订记录，变更可追溯 |

**集成方案的本质：给数字员工装一台"语义操作系统"——L1 是硬件事实，L2 是系统调用规范，L3 是编译器，L4 是进程。**

---


## 六、诚实边界：这份方案现在不是什么

遵守你自己定的写作红线（published ≠ 已实施）：

1. **业务组合编译器尚未实现**——ADR-005 的五职责与八字段是"已确认方向"，不是代码。本方案是**目标态架构**，不是实施清单。
2. **Design Artifact → Blueprint 的物化通道未定**——这正是 LangChat v2 战略与 BCM 的待对齐项（O-7/O-8 已 park，决策权归下游）。
3. **Agent Mapping 在 MI/BCM 侧无锚点**——D6 发现的缺口，W4 D5（数字员工定义）补。

**集成方案 v0.1 的价值不在于"已完成"，而在于：三套 ADR 从此有了一张共同的安装图纸，后续每个决策都能在这张图上找到坐标。**

---


## 七、与 W4 的衔接

明天进入 Week 4：**组装 MI CRE Enterprise Semantic Model v0.1（毕业作品）**。

本方案直接成为 W4 的骨架：


In [ ]:
W4 D1: 升级 Ontology Model     ← 补 L1 的语义命名缺口（W2 抽取结果）
W4 D2: 升级 Lifecycle + Event  ← 补 effect-registry 完整状态机
W4 D3: 补 Rule Model           ← 从代码 if-else 显式化
W4 D4: 升级 Capability + Policy ← 落实 P4 双层声明
W4 D5: Agent Mapping           ← 补 D6 发现的缺口
W4 D6: 组装 + 验证设计
W4 D7: Digital Employee Validation（"A101 为什么不能出租"六步验证）


---


## 八、架构师视角

**以前**：Ontology、Compiler、Agent 是三个轨道上的词——Ontology 像学术概念，Compiler 像工程模块，Agent 像产品功能，靠想象把它们连起来。

**现在**：三者是一条**语义供应链的四个工位**：世界模型（是什么）→ 语义治理（怎么组合）→ 编译（变成制品）→ 执行（受控行为）。架构师的核心工作从"设计组件"变成"守卫权威归位"——每一层只有一个权威、每一份产物只有一个出生地、每一次变更有一条升级路径（四态决策模型）。

---


## 九、练习（5 分钟）

用"商户减免审批"场景走一遍 3.2 的五阶段流水线：

1. 描述：这个能力涉及哪些对象？（提示：Contract / Billing / 减免单 / 审批流）
2. 治理：Ownership 谁是 canonical owner？Binding 是哪一类？
3. 组装：会产生什么 effect？前置条件是什么？
4. 物化：对应哪个 ApplicationContract？
5. 执行：数字员工做到哪一步必须停下来等人？

走不通的那一步，就是你下周 Semantic Model 要补的第一块砖。

---


## 附：本周交付物归档

| Day | 主题 | 核心产出 |
|-----|------|---------|
| D1 | Agent 如何利用 Ontology | Agent 认知地图（世界模型/规则/许可三供给） |
| D2 | Ontology Compiler | 编译器定位：BCM ADR-001/005 = Semantic Compiler 原则 |
| D3 | Ontology 约束 Agent 认知 | Context 边界 = 认知边界 |
| D4 | Policy 约束 Agent 执行 | 双层 Policy（语义允许 + 运行时强制） |
| D5 | Ontology 驱动 Digital Employee | 角色语义锚点（指向前，不包含） |
| D6 | 三套 ADR 在 Ontology 层统一 | 三层投影 + 五个语义锚点对齐表 |
| D7 | **集成方案 v0.1** | **四层架构 + 五阶段流水线 + 五个放置决策** |

*明日进入 Week 4：MI CRE Enterprise Semantic Model v0.1 组装与验证。*
